# 2. Preprocesamiento

Este notebook lee la tabla Delta creada en el notebook anterior, aplica transformaciones requeridas para entrenamiento y registra las variables generadas como features en un contexto de Feature Store.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName("control-2-preprocesamiento")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

bronze_df = spark.table("bronze_control_2")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

In [ ]:
feature_columns = [c for c in bronze_df.columns if c not in ("Outcome", "ingested_at")]

transformed_df = bronze_df.withColumn("Outcome", F.col("Outcome").cast("int"))
transformed_df = transformed_df.withColumn("processed_at", F.current_timestamp())

feature_sum_expr = F.lit(0.0)
for column_name in feature_columns:
    transformed_df = transformed_df.withColumn(f"{column_name}_norm", F.col(column_name) / F.lit(100.0))
    feature_sum_expr = feature_sum_expr + F.col(f"{column_name}_norm")

transformed_df = transformed_df.withColumn("feature_sum", feature_sum_expr)

transformed_df.printSchema()
transformed_df.show(5, truncate=False)

transformed_df.write.mode("overwrite").format("delta").saveAsTable("silver_control_2")
print("Datos procesados guardados en Delta como silver_control_2")

feature_df = transformed_df.select(
    *[f"{column_name}_norm" for column_name in feature_columns],
    "feature_sum",
    "Outcome"
)

try:
    from databricks.feature_store import FeatureStoreClient

    fs = FeatureStoreClient()
    fs.create_table(
        name="control_2_feature_store",
        primary_keys=["Outcome"],
        schema=feature_df.schema,
        description="Features generadas para el proyecto control-2",
        df=feature_df
    )
    print("Feature Store: tabla control_2_feature_store creada correctamente.")
except Exception as e:
    print(f"Feature Store no disponible en este entorno: {e}")
    feature_df.write.mode("overwrite").format("delta").saveAsTable("features_control_2")
    print("Se guardó una copia de features en Delta como features_control_2")